# 第8回B 演習: Forth インタプリタの実装

スライド: [第8回B](../08b-forth/)

スタック言語 Forth のインタプリタを、小さな部品を1つずつ埋めて作る。
土台は**第7回で作った逆ポーランド電卓** (`WORDS` / `evaluate`) で、
そこに「ワードを定義する仕組み」を足すと、電卓が言語になる。

```
3 4 + .          →  7
: SQUARE DUP * ;
5 SQUARE .       →  25
```

## 全体の地図

インタプリタは次の関数でできている。**★が自分で書く部分**で、他は用意してある。

```
forth_eval(stack, text)          用意済み: ワードを左から1つずつ取り出す
 ├─ read_definition(tokens, i)   ★課題3   ': 名前 本体 ;' を読む
 ├─ eval_user(stack, word)       ★課題3-b 自作ワードの本体を実行する
 └─ evaluate(stack, text)        ★課題1   第7回の電卓: WORDS を引いて実行する
                                 ★課題2   WORDS のワードを増やす (. DUP DROP SWAP OVER ROT)
repl(lines)                      ★課題4   1行ずつ評価する対話ループ
```

| 課題 | 書くもの | 配点 |
|---|---|---|
| 課題1 | `evaluate`, 四則演算 (第7回の課題4) | 20 |
| 課題2 | `.` とスタック操作ワード (第7回の課題4-b) | 20 |
| 課題3 | `read_definition` | 15 |
| 課題3-b | `eval_user` | 10 |
| 課題4 | `repl` | 15 |
| 発展1 | 比較演算子 | 10 |
| 発展2 | `IF / ELSE / THEN` | 10 |

## 進め方

- 上から順に進める。1つ埋めるたびに、直下の確認セルを実行する
- **課題1・2 は、第7回の課題4・4-b で書いたコードをそのまま貼ってよい** (名前が同じ)
- スタックは普通の Python のリスト (`append` が push、`pop` が pop)
- 確認セル (`# [CHECK]`) は自分用。消しても教員の採点は別途走るが、残すこと

---
## 課題0: 用意されたものを読む (採点なし)

次のセルを実行する。**この後の課題が埋める「枠」** がここにある。

- `WORDS`: 組み込みワードの表 (名前 → 関数)。第7回と同じ。課題1・2 で増やす
- `USER_WORDS`: 自分で定義したワードの表 (名前 → 本体のワードのリスト)
- `forth_eval`: ワードを1つずつ実行するループ。`:` なら定義を読み、自作ワードなら本体を実行し、
  それ以外は**第7回の `evaluate` に任せる**
- `run`: 空のスタックから実行して、できたスタックを返す (第7回の `calc_rpn` に相当)
- `trace`: 1語ごとにスタックの様子を表示する (うまく動かないときに使う)

In [ ]:
WORDS = {}          # 組み込みワード。第7回と同じ。課題1, 2 で増やす
USER_WORDS = {}     # 自作ワード。名前 -> 本体 (ワードのリスト)


def forth_eval(stack, text):
    """ワードを左から1つずつ実行する。text は文字列でもワードのリストでもよい"""
    tokens = text.split() if isinstance(text, str) else text
    i = 0
    while i < len(tokens):
        token = tokens[i]
        key = token.upper()
        if token == ':':
            name, body, i = read_definition(tokens, i)
            USER_WORDS[name] = body
            continue
        # 発展2: IF / ELSE / THEN の枝はここに足す
        if key in USER_WORDS:
            eval_user(stack, key)
        else:
            evaluate(stack, key)
        i += 1


def run(text):
    """空のスタックから text を実行して、できたスタックを返す"""
    stack = []
    forth_eval(stack, text)
    return stack


def trace(text):
    """1語ごとにスタックを表示する (デバッグ用。定義 : ... ; は含めないこと)"""
    stack = []
    for token in text.split():
        forth_eval(stack, [token])
        print(f'{token:>8}  {stack}')

In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '08b'


---
## 課題1: 電卓 — 第7回の課題4 (20点)

第7回の課題4 と同じ。**第7回で書いたコードがあれば、そのまま下のセルに貼ってよい。**

- `evaluate(stack, text)`: 空白で区切り、`WORDS` にあれば呼び、
  無ければ整数として push。整数でもなければ `ValueError`
- `+ - * /` を、関数を書いて `WORDS` に入れる。割り算は整数除算 `//`

**ヒント**: 引き算・割り算は**順序に注意**: 先に pop されるのは右側の値。
`token.lstrip('-').isdigit()` で負数も含めて数かどうか判定できる。

### 例題

次のセルを実行して、動きを見る。

`3 4 +` を手で追う。スタックは `append` で積み、`pop` で取る。

In [ ]:
# 例題
s = []
s.append(3)
s.append(4)
b = s.pop()
a = s.pop()
s.append(a + b)
print(s)    # [7]

s = [10, 3]
b, a = s.pop(), s.pop()
print(a - b)    # 7

`+` は書いてあるので、`- * /` を同じ形で足す。
`evaluate` は、第7回と同じく「`WORDS` にあれば呼ぶ・数なら積む・それ以外は `ValueError`」。

In [ ]:
def add(s):
    b, a = s.pop(), s.pop()
    s.append(a + b)


WORDS['+'] = add

# TODO: - * / も、+ と同じ形で関数を書いて WORDS に入れる (/ は整数除算 //)


def evaluate(stack, text):
    for token in text.split():
        # TODO: WORDS にあれば呼ぶ。なければ整数として積む。どちらでもなければ ValueError
        pass


print(run('3 4 +'))    # [7]
print(run('10 3 -'))   # [7]
trace('3 4 + 2 *')

In [ ]:
# [CHECK] q1
need('WORDS', kind=dict)
need('evaluate', arity=2)
need('forth_eval', arity=2)
need('run', arity=1)

assert run('3 4 +') == [7]
assert run('10 3 -') == [7]        # 順序に注意
assert run('3 4 + 2 *') == [14]
assert run('2 3 4 * +') == [14]
assert run('20 4 /') == [5]
assert run('1 2 3') == [1, 2, 3]
assert run('-5 3 +') == [-2]
assert run('') == []

# 性質で確かめる
holds('数を並べたらそのまま積まれる',
      lambda a, b: run(f'{a} {b}') == [a, b], ints(2, lo=0, hi=99))
holds('足してから引くと元の数',
      lambda a, b: run(f'{a} {b} + {b} -') == [a], ints(2, lo=0, hi=99))
try:
    run('3 FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')

# WORDS に後から足したワードも、evaluate を変えずに使えること
def _neg(s):
    s.append(-s.pop())
WORDS['NEG'] = _neg
assert run('5 NEG') == [-5], '実行中にワードを増やせること'
del WORDS['NEG']
print('OK')

---
## 課題2: `.` とスタック操作ワード — 第7回の課題4-b (20点)

関数を書いて `WORDS` に入れ、次のワードを登録せよ。**`evaluate` は変更しない**。
第7回の課題4-b で書いた分があれば貼ってよい。`.` だけが新しい。
名前は**大文字**で登録する (`forth_eval` が大文字にそろえてから渡すので、`dup` も動く)。

| ワード | 動作 | スタックの変化 |
|---|---|---|
| `.` | 一番上を**出力して**取り除く | (a → ) |
| `DUP` | 一番上を複製 | (a → a a) |
| `DROP` | 一番上を捨てる | (a → ) |
| `SWAP` | 上2つを入れ替え | (a b → b a) |
| `OVER` | 2番目を上にコピー | (a b → a b a) |
| `ROT` | 3番目を一番上へ | (a b c → b c a) |

**ヒント**: `(a b → b a)` は「左が実行前、右が実行後」。先に図を書いてからコードにする。
`DUP` は「上を覗いて積む」(`stack[-1]`)。`.` は取り出して `print`。

### 例題

次のセルを実行して、動きを見る。

ワードを1つ作って、`WORDS` に入れる形。

In [ ]:
# 例題
def neg(stack):
    stack.append(-stack.pop())


WORDS['NEG'] = neg
print(run('5 NEG'))    # [-5]
del WORDS['NEG']


In [ ]:
def dot(stack):
    # TODO: 一番上を取り出して print する
    pass


WORDS['.'] = dot


def dup(stack):
    # TODO
    pass


WORDS['DUP'] = dup


def drop(stack):
    # TODO
    pass


WORDS['DROP'] = drop


def swap(stack):
    # TODO
    pass


WORDS['SWAP'] = swap


def over(stack):
    # TODO
    pass


WORDS['OVER'] = over


def rot(stack):
    # TODO
    pass


WORDS['ROT'] = rot


print(run('1 2 SWAP'))    # [2, 1]
trace('1 2 3 ROT')


In [ ]:
# [CHECK] q2
need('WORDS', kind=dict)
need('forth_eval', arity=2)

import io, contextlib

assert run('1 2 SWAP') == [2, 1]
assert run('1 2 swap') == [2, 1]     # 小文字でも動くこと
assert run('5 DUP') == [5, 5]
assert run('1 2 DROP') == [1]
assert run('1 2 OVER') == [1, 2, 1]
assert run('1 2 3 ROT') == [2, 3, 1]
assert run('5 dup *') == [25]
assert run('3 4 + 2 *') == [14]      # 課題1 が壊れていないこと
assert run('10 3 -') == [7]

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    s = run('42 .')
assert s == [], 'ドットは取り除くこと'
assert '42' in buf.getvalue(), 'ドットは出力すること'

try:
    run('FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')
print('OK')

---
## 課題3: ワードの定義を読む (15点)

`: 名前 本体 ;` を読み取る `read_definition(tokens, i)` を書け。
`tokens` はワードのリスト、`tokens[i]` が `:` である。返すのは次の3つ組。

```
(名前を大文字にしたもの, 本体のワードのリスト, ';' の次の位置)
```

```
tokens = [':', 'SQUARE', 'DUP', '*', ';', '5', 'SQUARE']
添字      0      1        2     3    4    5     6
read_definition(tokens, 0)  →  ('SQUARE', ['DUP', '*'], 5)
```

- `;` が無い、または名前が無い (`: ;`) なら `ValueError`
- 定義を**実行はしない**。読み取るだけ (登録は用意済みの `forth_eval` がする)

**ヒント**: `tokens.index(';', i)` は、位置 `i` から先で最初の `;` の添字を返す。
無ければ `ValueError` を出す。名前は `tokens[i + 1]`、本体は `tokens[i + 2 : ';' の位置]` のスライス。

### 例題

次のセルを実行して、動きを見る。

`;` の位置を探して、スライスで切り出す。

In [ ]:
# 例題
tokens = ': SQUARE DUP * ; 5 SQUARE'.split()
j = tokens.index(';', 0)
print(j)               # 4
print(tokens[1])       # SQUARE
print(tokens[2:j])     # ['DUP', '*']
print(j + 1)           # 5

In [ ]:
def read_definition(tokens, i):
    # TODO
    pass


print(read_definition(': SQUARE DUP * ; 5 SQUARE'.split(), 0))   # ('SQUARE', ['DUP', '*'], 5)

In [ ]:
# [CHECK] q3
need('read_definition', arity=2)

assert read_definition(': SQUARE DUP * ; 5'.split(), 0) == ('SQUARE', ['DUP', '*'], 5)
assert read_definition('1 : inc 1 + ; inc'.split(), 1) == ('INC', ['1', '+'], 6), '名前は大文字にすること'
assert read_definition(': NOP ;'.split(), 0) == ('NOP', [], 3), '本体が空でもよい'
assert read_definition(': A B ; : C D ;'.split(), 4) == ('C', ['D'], 8), '2つ目の定義'

for bad in (': FOO DUP', ':', ': ;'):
    try:
        read_definition(bad.split(), 0)
    except ValueError:
        pass
    else:
        raise AssertionError(f'{bad!r} は ValueError にすること')
print('OK')

---
## 課題3-b: 自作ワードを実行する (10点)

用意済みの `forth_eval` は、`:` を見つけると `read_definition` で読み、
`USER_WORDS[名前] = 本体` に入れる。ここで **`SQUARE` のような自作ワードが呼ばれたとき**に
本体を実行する `eval_user(stack, word)` を書け。

```
: DOUBLE DUP + ;
: QUADRUPLE DOUBLE DOUBLE ;
5 QUADRUPLE .       →  20
```

**ヒント**: 本体は `USER_WORDS[word]` にあるワードのリスト。
`forth_eval` は**ワードのリストも受け取れる**ので、それを渡して呼ぶだけ。
`QUADRUPLE` が `DOUBLE` を呼べるのは、`forth_eval` が自分自身を再び呼ぶ形になっているから。

### 例題

次のセルを実行して、動きを見る。

リストを渡しても `forth_eval` は動く。

In [ ]:
# 例題
s = [5]
forth_eval(s, ['DUP', '*'])
print(s)    # [25]

In [ ]:
def eval_user(stack, word):
    # TODO
    pass


USER_WORDS.clear()
print(run(': SQUARE DUP * ;  5 SQUARE'))                                  # [25]
print(run(': DOUBLE DUP + ; : QUADRUPLE DOUBLE DOUBLE ;  5 QUADRUPLE'))   # [20]

In [ ]:
# [CHECK] q3b
need('USER_WORDS', kind=dict)
need('eval_user', arity=2)
need('forth_eval', arity=2)

USER_WORDS.clear(); assert run(': SQUARE DUP * ; 5 SQUARE') == [25]
USER_WORDS.clear(); assert run(': DOUBLE DUP + ; : QUAD DOUBLE DOUBLE ; 5 QUAD') == [20]
USER_WORDS.clear(); assert run(': INC 1 + ; 10 INC INC INC') == [13]
USER_WORDS.clear(); assert run('1 : ADD2 2 + ; ADD2 ADD2') == [5]
USER_WORDS.clear(); assert run(': SQ DUP * ; 3 SQ SQ') == [81]

# 定義しただけでは何もスタックに積まれない
USER_WORDS.clear(); assert run(': NOP ;') == []

# 大文字小文字を区別しないこと
USER_WORDS.clear(); assert run(': SQUARE DUP * ; 4 square') == [16]

# 組み込みが壊れていないこと
USER_WORDS.clear()
assert run('3 4 + 2 *') == [14]
assert run('1 2 3 ROT') == [2, 3, 1]
# ユーザ定義は USER_WORDS に入れ、組み込みの表は汚さないこと
USER_WORDS.clear()
_before = set(WORDS)
run(': SQUARE DUP * ;')
assert set(WORDS) == _before, 'ユーザ定義を WORDS に混ぜないこと'
assert len(USER_WORDS) == 1, f'USER_WORDS が {len(USER_WORDS)} 件'
note('ユーザ定義ワードは組み込みと別の表に入っている')

print('OK')

---
## 課題4: 対話ループ (REPL) (15点)

1行ずつ評価して、結果を返すループを書け。
**採点できるように、`input()` ではなく引数で入力を受け取る形にする。**

`repl(lines)` を書け。`lines` は入力行のリスト。
返り値は**各行に対する出力文字列のリスト**。

| 入力行 | 返すもの |
|---|---|
| 普通の行 | 評価後のスタックを `str()` にしたもの (例: `'[7]'`) |
| `bye` | そこで終了。それ以降の行は処理しない |
| エラーが起きた行 | `'error: ' + エラーメッセージ` を返し、**ループは続ける** |

スタックは行をまたいで保たれること (1行目で積んだ値を2行目で使える)。

**ヒント**: 第7回の例外処理。`except Exception as e:` で捕まえて `str(e)` を使う。
`stack` は**ループの外**で作る。

### 例題

次のセルを実行して、動きを見る。

1行ずつ、失敗しても続ける形。

In [ ]:
# 例題
outputs = []
stack = []
for line in ['3 4 +', 'FOO']:
    try:
        forth_eval(stack, line)
        outputs.append(str(stack))
    except Exception as e:
        outputs.append('error: ' + str(e))
print(outputs)    # ['[7]', 'error: ...']

`bye` は `try` に入る前に判定して `break`。
`stack` をループの外で作っているので、値は行をまたいで残る。


In [ ]:
def repl(lines):
    stack = []
    outputs = []
    for line in lines:
        # TODO: bye なら break / 評価して str(stack) を足す / エラーなら 'error: ...' を足す
        pass
    return outputs


USER_WORDS.clear()
for out in repl(['3 4 +', '2 *', 'FOO', ': SQ DUP * ;', 'SQ', 'bye', '999']):
    print(out)

In [ ]:
# 対話用 (採点しない)。実行すると入力待ち
def repl_interactive():
    stack = []
    while True:
        line = input('> ')
        if line.strip().lower() == 'bye':
            break
        try:
            forth_eval(stack, line)
            print(stack)
        except Exception as e:
            print('error:', e)


# repl_interactive()   ← コメントを外すと対話できる

In [ ]:
# [CHECK] q4
need('repl', arity=1)

USER_WORDS.clear()
out = repl(['3 4 +', '2 *'])
assert out == ['[7]', '[14]'], out

# bye で止まり、それ以降は処理しない
USER_WORDS.clear()
out = repl(['1 2 +', 'bye', '999'])
assert out == ['[3]'], out

# エラーでもループは続く
USER_WORDS.clear()
out = repl(['1 2 +', 'FOO', '10 +'])
assert len(out) == 3, out
assert out[0] == '[3]'
assert out[1].startswith('error:'), out[1]
assert out[2] == '[13]', 'エラーのあともスタックが保たれること'

# ユーザ定義ワードも行をまたいで残る
USER_WORDS.clear()
out = repl([': SQ DUP * ;', '5 SQ'])
assert out == ['[]', '[25]'], out

assert repl([]) == []
assert repl(['bye']) == []
print('OK')


---
# 発展課題

余力があれば。


## 発展1: 比較演算子 (10点)

`<` `>` `=` を `WORDS` に追加せよ。Forth では**真が `-1`、偽が `0`** で表される
(歴史的な理由: すべてのビットが 1)。

### 例題

次のセルを実行して、動きを見る。

比較の結果を Forth の真偽値に直す。

In [ ]:
# 例題
def less(stack):
    b, a = stack.pop(), stack.pop()
    stack.append(-1 if a < b else 0)


WORDS['<'] = less


`>` と `=` も同じ形。Python の `True` を積むと `1` になってしまうので、
`-1` / `0` に変換すること。


In [ ]:
# TODO: > と = も、< と同じ形で関数を書いて WORDS に入れる


In [ ]:
# [CHECK] b1
USER_WORDS.clear()
assert run('3 4 <') == [-1]
assert run('4 3 <') == [0]
assert run('3 4 >') == [0]
assert run('4 3 >') == [-1]
assert run('4 4 =') == [-1]
assert run('3 4 =') == [0]
assert run('4 4 <') == [0] and run('4 4 >') == [0]
assert run('1 2 + 3 =') == [-1]
print('OK')


## 発展2: 条件分岐 (10点)

`IF ... ELSE ... THEN` を実装せよ。`IF` の直前にスタックにある値が
0 以外なら `IF` と `ELSE` の間を、0 なら `ELSE` と `THEN` の間を実行する
(`ELSE` は省略できる)。

```
: ABS DUP 0 < IF 0 SWAP - THEN ;
-5 ABS .     →  5
```

**どこを直すか**: 課題0 のセルの `forth_eval`。`:` の枝のすぐ下に
`# 発展2` の目印がある。そこに `IF` の枝を足す (`:` の枝と同じく、`i` を進めて `continue`)。

**ヒント**: 課題3 と同じく、トークン列の上で対応する `ELSE` / `THEN` を探す。
`IF` が入れ子になりうることに注意 (深さを数える)。

### 例題

次のセルを実行して、動きを見る。

入れ子を数えながら、対応する `THEN` を探す。

In [ ]:
# 例題
def find_then(tokens, i):
    """tokens[i] == 'IF' のとき、対応する THEN の位置を返す"""
    depth = 1
    j = i + 1
    while j < len(tokens):
        if tokens[j].upper() == 'IF':
            depth += 1
        elif tokens[j].upper() == 'THEN':
            depth -= 1
            if depth == 0:
                return j
        j += 1
    raise ValueError('THEN がない')

`ELSE` も同じ要領で、`depth == 1` のものだけを自分のものとして拾う。
条件は `IF` に来た時点で `stack.pop()`。実行する側のトークン列を切り出して
`forth_eval(stack, 切り出した部分)` に渡し、`i` は `THEN` の位置まで飛ばす。


In [ ]:
# TODO: 課題0 のセルの forth_eval に IF / ELSE / THEN を足す (このセルは空のままでよい)

In [ ]:
# [CHECK] b2
USER_WORDS.clear()
assert run('1 IF 42 THEN') == [42]
assert run('0 IF 42 THEN') == []
assert run('1 IF 10 ELSE 20 THEN') == [10]
assert run('0 IF 10 ELSE 20 THEN') == [20]
assert run('-1 IF 7 THEN') == [7], '0 以外はすべて真'

USER_WORDS.clear()
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; -5 ABS') == [5]
USER_WORDS.clear()
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; 5 ABS') == [5]

# 入れ子
USER_WORDS.clear()
assert run('1 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [7]
assert run('1 IF 0 IF 7 ELSE 8 THEN ELSE 9 THEN') == [8]
assert run('0 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [9]

# 条件分岐の外側は普通に続く
assert run('1 IF 10 THEN 5 +') == [15]
print('OK')


---
## 自己確認

`score()` で全部まとめて確かめる。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。


In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` と採点ページは下書き確認用。

1. 上から実行し、出力を残す
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても教員の採点は別の検査で行う。
